In [5]:
import pandas as pd

perf1 = pd.read_csv('Schubert_D911-03_measure_to_audio_time.csv', sep=',')
perf2 = pd.read_csv('Schubert_D911-03_measure_to_audio_time2.csv', sep=',')

df = pd.merge(perf1, perf2, on='measure', suffixes=('_HU33', '_SC06'))
df = df.rename(columns={'audio_time_in_seconds_HU33': 'time_HU33', 'audio_time_in_seconds_SC06': 'time_SC06'})

# Test code: In column "time_score", i want the runtime in seconds at this measure position. Consider tempo (BPM), time signatures etc.
# bpm = 60
# time_signature = (2, 2)
# seconds_per_measure = time_signature[0] * (2 / time_signature[1]) * 60 / bpm
# df['time_score'] = df['measure'] * seconds_per_measure
df = df[['measure', 'time_score', 'time_HU33', 'time_SC06']]

print(df)
df.to_csv('alignment.csv', index=False)

      measure  time_score  time_HU33  time_SC06
0        0.00        0.00       0.00       0.00
1        0.02        0.04       0.02       0.02
2        0.04        0.08       0.04       0.04
3        0.06        0.12       0.06       0.06
4        0.08        0.16       0.08       0.08
...       ...         ...        ...        ...
2769    55.40      110.80     143.30     135.54
2770    55.42      110.84     143.32     135.56
2771    55.44      110.88     143.34     135.58
2772    55.46      110.92     143.36     135.60
2773    55.48      110.96     150.34     137.18

[2774 rows x 4 columns]


In [8]:
import math
import pandas as pd
from music21 import converter, meter, repeat

# ------------------------------------------------------------------
# Load performance alignments
# ------------------------------------------------------------------

perf1 = pd.read_csv('Schubert_D911-03_measure_to_audio_time.csv', sep=',')
perf2 = pd.read_csv('Schubert_D911-03_measure_to_audio_time2.csv', sep=',')

df = pd.merge(perf1, perf2, on='measure', suffixes=('_HU33', '_SC06'))
df = df.rename(columns={
    'audio_time_in_seconds_HU33': 'time_HU33',
    'audio_time_in_seconds_SC06': 'time_SC06'
})

# ------------------------------------------------------------------
# Score -> theoretical runtime mapping
# ------------------------------------------------------------------

score_path = 'Schubert_D911-03.musicxml'   # change this if needed


def load_score(xml_path, expand_repeats=False):
    score = converter.parse(xml_path)
    if expand_repeats:
        try:
            score = repeat.Expander(score).process()
        except Exception:
            pass
    return score


def get_reference_part(score):
    return score.parts[0] if hasattr(score, 'parts') and len(score.parts) > 0 else score


def build_tempo_segments(score, default_bpm=120.0):
    """
    Build a piecewise-linear mapping from score offset (quarter lengths)
    to theoretical runtime (seconds).
    """
    boundaries = score.metronomeMarkBoundaries()

    if not boundaries:
        total_len = float(score.highestTime)
        return [{
            'start_q': 0.0,
            'end_q': total_len,
            'bpm': float(default_bpm),
            'sec_at_start': 0.0
        }]

    segments = []
    sec_acc = 0.0

    for start, end, mm in boundaries:
        bpm = None

        if mm is not None:
            try:
                bpm = float(mm.getQuarterBPM())
            except Exception:
                pass

            if bpm is None:
                try:
                    bpm = float(mm.number)
                except Exception:
                    pass

        if bpm is None or bpm <= 0:
            bpm = float(default_bpm)

        start = float(start)
        end = float(end)

        segments.append({
            'start_q': start,
            'end_q': end,
            'bpm': bpm,
            'sec_at_start': sec_acc
        })

        sec_acc += (end - start) * 60.0 / bpm

    return segments


def offset_to_seconds(offset_q, tempo_segments):
    offset_q = float(offset_q)

    for seg in tempo_segments:
        if seg['start_q'] <= offset_q < seg['end_q']:
            return seg['sec_at_start'] + (offset_q - seg['start_q']) * 60.0 / seg['bpm']

    # Allow exact end of piece
    if tempo_segments and math.isclose(offset_q, tempo_segments[-1]['end_q']):
        seg = tempo_segments[-1]
        return seg['sec_at_start'] + (seg['end_q'] - seg['start_q']) * 60.0 / seg['bpm']

    raise ValueError(f'Offset {offset_q} outside score timeline.')


def build_measure_map(part):
    """
    One row per measure:
      - measure number (can be 0 for pickup / prelude bar)
      - start offset in quarter lengths
      - actual duration of that measure in quarter lengths

    Important:
    For pickup bars, we use the actual measure duration, not the nominal
    full-bar duration from the time signature.
    """
    rows = []
    current_ts = None

    for meas in part.getElementsByClass('Measure'):
        ts = meas.timeSignature or current_ts
        if ts is None:
            ts = meas.getContextByClass(meter.TimeSignature)
        if ts is not None:
            current_ts = ts

        if current_ts is None:
            raise ValueError(f'No time signature available at measure {meas.measureNumber}.')

        rows.append({
            'measure_number': int(meas.measureNumber),
            'start_q': float(meas.offset),
            'measure_duration_q': float(meas.duration.quarterLength),
            'time_signature': current_ts.ratioString
        })

    measure_map = pd.DataFrame(rows).sort_values(['measure_number', 'start_q']).reset_index(drop=True)
    return measure_map


def fractional_measure_to_offset(frac_measure, measure_map):
    """
    Convert a fractional measure number to a global score offset.

    Examples:
      1.0  -> start of measure 1
      1.5  -> halfway through measure 1
      0.75 -> 75% through measure 0 (pickup/prelude), if measure 0 exists

    Fraction is interpreted relative to the actual duration of that measure.
    """
    frac_measure = float(frac_measure)

    measure_num = math.floor(frac_measure)
    frac_inside_measure = frac_measure - measure_num

    row = measure_map.loc[measure_map['measure_number'] == measure_num]
    if row.empty:
        raise ValueError(f'Measure {measure_num} not found in score.')
    row = row.iloc[0]

    return row['start_q'] + frac_inside_measure * row['measure_duration_q']


def measure_to_score_time(frac_measure, measure_map, tempo_segments):
    offset_q = fractional_measure_to_offset(frac_measure, measure_map)
    return offset_to_seconds(offset_q, tempo_segments)


# ------------------------------------------------------------------
# Parse score and compute theoretical score time
# ------------------------------------------------------------------

score_path = "example_dtw/data/Schubert_D911-03.xml"
score = load_score(score_path, expand_repeats=False)
ref_part = get_reference_part(score)

measure_map = build_measure_map(ref_part)
tempo_segments = build_tempo_segments(score, default_bpm=120.0)

df['time_score'] = df['measure'].apply(
    lambda x: measure_to_score_time(x, measure_map, tempo_segments)
)

# Reorder columns
df = df[['measure', 'time_score', 'time_HU33', 'time_SC06']]

print(df)
df.to_csv('alignment.csv', index=False)

/home/manuel/trackswitch.js/.venv/lib/python3.11/site-packages/music21/musicxml/xmlToM21.py:5021: MusicXMLWarning: Cannot put in an element with a missing voice tag when no previous voice tag was given.  Assuming voice 1... 
  warnings.warn('Cannot put in an element with a missing voice tag when '
/home/manuel/trackswitch.js/.venv/lib/python3.11/site-packages/music21/musicxml/xmlToM21.py:5034: MusicXMLWarning: Cannot find voice 1; putting outside of voices.
  warnings.warn(
/home/manuel/trackswitch.js/.venv/lib/python3.11/site-packages/music21/musicxml/xmlToM21.py:5037: MusicXMLWarning: Current voiceIds: ['3', '4']
  warnings.warn(
/home/manuel/trackswitch.js/.venv/lib/python3.11/site-packages/music21/musicxml/xmlToM21.py:5040: MusicXMLWarning: Current voices: [<music21.stream.Voice 3>, <music21.stream.Voice 4>] in m. 20
  warnings.warn(
/home/manuel/trackswitch.js/.venv/lib/python3.11/site-packages/music21/musicxml/xmlToM21.py:5040: MusicXMLWarning: Current voices: [<music21.stream.Vo

      measure  time_score  time_HU33  time_SC06
0        0.00        0.00       0.00       0.00
1        0.02        0.02       0.02       0.02
2        0.04        0.04       0.04       0.04
3        0.06        0.06       0.06       0.06
4        0.08        0.08       0.08       0.08
...       ...         ...        ...        ...
2769    55.40      218.60     143.30     135.54
2770    55.42      218.68     143.32     135.56
2771    55.44      218.76     143.34     135.58
2772    55.46      218.84     143.36     135.60
2773    55.48      218.92     150.34     137.18

[2774 rows x 4 columns]
